# <span style = 'color: blue;'>Full Mojito Spectrogram </span>
### A spectrogram is essentially a 2d heatmap with frequency on one axis (y in this case) and time on another, where the color represents signal strength. A time series (amplitude as a function of time) is used to get the periodogram (amplitude as a function of frequency) (via a Fourier Transform), which shows which frequencies have the greatest strength and are useful for identifying the strongest periodic behavior in time series data. However, the more robust version of a periodogram is the power spectral density (PSD), which is plotted as amplitude divided by frequency as a function of frequency. A spectrogram shows the most information: it breaks the data up into chunks and calculates the PSD, which demonstrates how the PSD changes over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy.signal import welch, windows, periodogram

import warnings
warnings.simplefilter("always")

import matplotlib.style as mplstyle
mplstyle.use('fast')


## Loading Signal and Noise Data

In [ ]:
f = h5py.File("mojito_light_731d_2.5s_L1_0_0_20251218T000909441368Z.h5",'r')
f2 = h5py.File('MBHB_731d_2.5s_L1_source3_0_20251203T230844045311Z.h5', 'r')
dt = f['tdis/sampling'].attrs['dt']
T = f['tdis/sampling'].attrs['duration']
dtSingle = f2['tdis/sampling'].attrs['dt']  # time per sample (confirmed by fs)
TSingle = f2['tdis/sampling'].attrs['duration']  # total time taken for the data (confirmed by Ira, also it's around 2pi * 10^7 which is 2 years which we know is the duration)
fsSingle = 1/dtSingle
fs = 1/dt  # sampling frequency (confirmed by welch())
ASingle = np.array(f2['tdis/A2'])  # TDI A variable
A = np.array(f['tdis/A2'])


In [ ]:
fest_min  = f['noise_estimates/log_frequency_sampling'].attrs['fmin']
fest_max  = f['noise_estimates/log_frequency_sampling'].attrs['fmax']
fest_size = f['noise_estimates/log_frequency_sampling'].attrs['size']
fest = np.exp(np.linspace(np.log(fest_min), np.log(fest_max), fest_size))  # estimated sampling frequencies for the PSD

print(fest_min, fest_max)

est_psdA = np.real(np.array(f['noise_estimates/AET'][0, :, 0, 0]))  # estimated power spectral density of the noise


## Getting PSD curves for Estimated Signal and Noise Data

In [ ]:
# Welch PSD---------------
Nsegs = 100
fw, Aw = welch(A,fs,noverlap=0,nperseg=len(A)/Nsegs, window='hann')  # scipy.signal.welch() takes in a time series (A) and sampling frequency (fs) and returns the power spectral density (PSD) of the time series, as in basically the amplitude as a function of frequency. So, it returns an array of sample frequencies and an array of the PSD values. The "w"s in the variables indicate "welch".
fig1, axs1 = plt.subplots(1, 2, figsize = (12, 5))

#axs1[0].loglog(fw,Aw, label = 'Signal')  # plotting the PSD of the GW signal ?
axs1[0].loglog(fest, est_psdA, label = 'Noise')  # plotting the noise?
axs1[0].set_title('Signal and Noise PSDs (Welch)')


# Periodogram-------------
fp, Ap = periodogram(A,fs,'hann')  # scipy.signal.periodogram also takes in a time series and sampling frequency and returns the PSD, but it uses the whole data altogether unlike welch() which splits up the data and calculates periodograms for each section.

#axs1[1].loglog(fp, Ap)
axs1[1].loglog(fest, est_psdA)
axs1[1].set_title('Signal and Noise Periodograms')


# Figure Addendum---------
fig1.supxlabel('Frequency')
fig1.supylabel('PSD')
fig1.suptitle('Welch PSD vs Periodogram Comparison')
fig1.legend()
fig1.tight_layout()

plt.savefig('Actual Noise.jpg')


We can see that the signal-to-noise ratio (SNR) is better at lower frequencies, and that the Welch signal looks cleaner.

## Making a Spectrogram (Robbie's Method)

In [ ]:
from WDMWaveletTransforms.wavelet_transforms import (
    transform_wavelet_freq,
    transform_wavelet_freq_time,
    transform_wavelet_time,
)


In [ ]:
WAVELET_DURATION = 3600*2  # 2 hours worth of seconds
ND = int(T / dt)  # total number of data points. (also, better to do int() than // bc we might need int for making arrays of data)
NT = int(T) // WAVELET_DURATION  #  how many full wavelets did we record ? which is also the number of time data points ? (also, better to do int() than // bc we might need int for making arrays of data)

# Force the number of wavelets to be even for some reason ?
if NT%2 >0:
    NT -= 1

NF = ND // NT  # a wavelet is a range of times longer than the sampling time (1/sampling frequency), so it contains many data points. In our case, the sampling time is 2.5s, the wavelet duration is 2 hours, and so the number of data points per wavelet is 2 hours / 2.5 s = 2880. If you also assign one time to each wavelet, say the time in the middle of the wavelet, you'll have a number of times equal to the number of wavelets. Each wavelet will have 2880 data points, and that's this value: number of data points over number of wavelets = number of data points per wavelet. We can do WAVELET_DURATION / dt and it's the same value as ND / NT. Still confused on why this is the number of frequencies, unless it's just since each wavelet has a set number of times, each one corresponds to a different frequency (like, maybe t within the wavelet minus t at the start of the wavelet, so like the 3rd data point in the wavelet would have frequency 1/(3 * 2.5 s)?) Or actually, maybe it's because we need a single frequency for each data point in the wavelet so we can do calculations, and we already have that many times per wavelet (just 0, 2.5 s, 5 s, etc)?

# Force the ratio to be even for some reason ?
if NF%2 > 0:
    NF -=1

ND = NF*NT  # NF is defined as ND // NT so because of the floor div it can cause discrepancies, which are solved by doing this. However, this does make ND different from the total number of data points.

print(f'NT (number of wavelet durations recorded): {NT}')
print(f'NF (number of frequencies): {NF}')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
from matplotlib import colors
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
#import seaborn as sns

def wavelet_specgram(t, f, pz, fmin = 0, fmax = np.inf, tmin=0, tmax= np.inf, cmap = 'bwr', vmin=None,vmax=None,log=False,filename=None, logf=False):
    Nt = len(t)  # number of times
    Nf = len(f)  # number of frequencies
    if tmax < np.inf:
        et = np.argmin(np.abs(t-tmax))
    else:
        et = Nt-1
    if fmax < np.inf:
        e = np.argmin(np.abs(f-fmax))
    else:
        e = Nf-1
    emin = np.argmin(np.abs(f-fmin))
    etmin = np.argmin(np.abs(t-tmin))
    if pz.size != (Nt-1)*(Nf-1):
        print(pz.shape,Nt,Nf)
        print("Nt or Nf is wrong")
    else:
        print(f'Nt of Nf is not wrong. Nt: {Nt}; Nf: {Nf}; pz shape: {pz.shape}')
    print(f'Setting z = pz[{emin}:{e}, {etmin}:{et}].copy().')
    z = pz[emin:e,etmin:et].copy()
    z /= z.max()
    print(f'z shape: {z.shape}')

    if vmin == None:
        vmin=z.min()+1e-20
    if vmax == None:
        vmax=z.max()
    if log:
        norm = colors.LogNorm(vmin = vmin, vmax=vmax)
    else:
        norm = colors.Normalize(vmin = vmin, vmax = vmax)

    z_clean = np.nan_to_num(z, nan=z.min(), posinf=z.min(), neginf=z.min())

    maxarr=[]
    fig, ax = plt.subplots(figsize=(10,7))
    #plt.pcolormesh(t, f[:e+1], z_clean.reshape((Nt-1,e)).T,cmap=cmap,norm=norm,shading='auto',snap=False, rasterized=True)
    plt.pcolormesh(t[etmin:et+1], f[emin:e+1], z_clean.reshape((e-emin,et-etmin)),cmap=cmap,norm=norm,shading='auto',snap=False, rasterized=True)  # reshaping like that literally never does anything
    tmesh, fmesh = np.meshgrid(t[etmin:et + 1], f[emin: e + 1])
    #ax.scatter(tmesh.flat, fmesh.flat, marker = 'o', color = 'lime', s = 2, rasterized = True)

    #plt.contourf(t, f[:e+1], z_clean.reshape((e,Nt-1)), levels=100, cmap=cmap,norm=norm,shading='auto',snap=False, rasterized=True)
    plt.grid(False)
    plt.xlabel('t (sec)')
    plt.ylabel('f (Hz)')
    if logf:
        ax.set_yscale('log')
    if filename:
        plt.savefig(filename)
    plt.show()
    return np.real(t[etmin:et+1]), np.real(f[emin:e+1]), z_clean.reshape((e-emin,et-etmin))

def scalogram(data,dt,Nt=128//1,Nf=None, colormap = 'bwr', vmin=None,log=False,logf = False, filename=None,fmin = 0, fmax=np.inf, tmin=0, tmax = np.inf, whiten_PSD=None, whiten_f = None, overlay_data = None):  # i guess scalogram stands for scaled spectrogram ?
    if not Nf:
        Nf = data.shape[0]//Nt  # number of frequencies also you can just do len(data) since the data is the A, E, or T and they're just 1D arrays.
    ND = Nf*Nt  # number of data points
    Tobs = ND*dt  # total observation duration
    wt = np.linspace(0,Tobs,Nt+1)  # a time array from 0 to the total observation time, containing Nt + 1 elements

    wf = np.arange(0,Nf+1)*1/(2*dt*Nf)  # a frequency array from 0 to Nf + 1, going up to the sampling rate over 2 (the Nyquist frequency)
    """
    0, 1, 2, 3, ..., Nf + 1
    -->
    0/(Nf + 1), 1/(Nf + 1), ..., (Nf + 1)/(Nf + 1)
    = # let's say Nf = 9, for example
    0, 0.1, 0.2, ..., 1  # it'll always be numbers between 0 and 1
    -->
    0/2.5, 0.1/2.5, ..., 1/2.5  # converting the values to frequencies; 1/2.5 is the sampling frequency
    -->
    0/2.5/2, 0.1/2.5/2, ..., 1/2.5/2  # dividing every frequency by 2 so that the max frequency is the Nyquist frequency, which is half the sampling rate (1/2.5/2)
    """

    print(f'ND: {ND}')
    print(f'Data length: {len(data)}')
    dataf = np.fft.fft(data[:ND]*windows.tukey(ND,0.01))  # first, only grab the first ND elements from the data just in case there's more data than ND. Then, multiply by a Tukey window (which is a tapered
    aa = transform_wavelet_freq(dataf,Nf,Nt).T
    #complex_coeff = aa[:, ::2] + 1j * aa[:, 1::2]
    wt=wt[::2]
    aa = np.abs(aa[:,::2])
    #aa = np.sqrt(aa[:,1::2]**2+aa[:,::2]**2)
    #bb = np.zeros_like(aa)
    #if overlay_data is not None:
    #    overlay_dataf = np.fft.fft(overlay_data[:ND]*windows.tukey(ND,0.01))
    #    cc = transform_wavelet_freq(dataf,Nf,Nt).T
    #    bb = np.abs(cc[:,::2])
    if whiten_f is not None:
        interp_whiten_PSD = np.interp(wf[1:], whiten_f, whiten_PSD)
        aa /= np.sqrt(interp_whiten_PSD[:,np.newaxis])
    #    bb /= np.sqrt(interp_whiten_PSD[:,np.newaxis])
    arr = wavelet_specgram(wt, wf, aa,fmin=fmin, fmax=fmax, tmin=tmin, tmax=tmax, vmin=vmin,log=log,logf=logf,filename=filename, cmap = colormap)#, overlay=(overlay_data is not None), overlay_wPSD = bb)
    return arr



In [ ]:
wf = np.arange(0,NF+1)*1/(2*dt*NF)
whiten_f = fest
whiten_PSD = est_psdA

interp_whiten_PSD = np.interp(wf[1:], whiten_f, whiten_PSD)
#print(interp_whiten_PSD)

aa = np.arange((len(wf) - 1) * 4, dtype = np.float64).reshape(len(wf) - 1, 4)
print(f'aa: {aa}')
thing = 1/np.sqrt(interp_whiten_PSD[:, np.newaxis])
print(f'PSD: {thing}')

aa *= thing
print(f'aa now: {aa}')

print()
print(f'aa.shape: {aa.shape}; thing.shape: {thing.shape}')

In [ ]:
wt, wf, wPSD = scalogram(A, dt, Nt=NT, Nf=NF, whiten_f = fest, whiten_PSD=est_psdA, log=True, logf=True, fmin = 1e-4, fmax = 1e-2, filename="mojito_sgram10.png", colormap = 'inferno')


In [ ]:
wavelet_specgram(wt, wf, wPSD, fmin=1e-4, fmax=1e-2,log=True,logf=True,filename="mojito_website.png", vmin=1e-4, vmax = 2e-1, cmap='inferno')

simulate 3 months waveform, and then at the ends, we use a taper function to smoothly change the value to either 0 or some constant offset. this can be a cosine function. at the beginning of the waveform, it'll be pretty close to 0, but at the end, there's a "gravitational wave memory effect" that can create the offset. basically, the 3 month data endpoints are not going to be exactly 0, so we need a way to smoothly get to 0 (or the offset) at each end.

to determine the time window, we can see if the curve of the freq. vs. time data gets dim near the left side of the time window and if there's enough space between the high curve and the right edge of the time window.

X, Y, and Z are the responses of the LISA detector. but these are transformations of h, which is the actual signal of the gravitational wave.

we can use one of the 3 pytorch pretrained cnns to fine tune (faster training) or create a custom network maybe based on mask-r-cnn (slower training)

## Making a Spectrogram (My way)

In [ ]:
import time
from WDMWaveletTransforms.wavelet_transforms import transform_wavelet_freq
from matplotlib import colors


def waveletSpectrogram(dataList, dt, Nt = 128//1, Nf = None, cmap = 'inferno', vmin = None, vmax = None, logColor = True, logFreq = True, fmin = 0, fmax = np.inf, noiseLevel = 1e-24, whiten_PSD = None, whiten_f = None, overlay_data = None, filename = None, plotFeatures = False):
    # Get the time and frequency coordinates for the pcolormesh
    if Nf is None:  # if the number of frequency data points isn't given, set it to what it should be
        Nf = len(data) // Nt
    ND = Nf * Nt  # number of data points
    TObs = ND * dt  # total observation duration
    tCoords = np.linspace(0, TObs, Nt + 1)  # time array from 0 to TObs, containing Nt + 1 elements to denote the x-values of the corners of the plt.pcolormesh rectangles.

    fCoords = np.arange(0, Nf + 1)*1/(2*dt*(Nf))  # a frequency array with Nf + 1 elements (to denote the y-values of the corners of the plt.pcolormesh rectangles), going up to just under the sampling rate over 2 (the Nyquist frequency)
    """ Worked example showing how fCoords is made
    0, 1, 2, 3, ..., Nf
    -->
    0/Nf, 1/Nf, ..., Nf/Nf
    = # let's say Nf = 10, for example
    0, 0.1, 0.2, ..., 1  # it'll always be from 0 to 1
    -->
    0/2.5, 0.1/2.5, ..., 1/2.5  # converting the values to frequencies; 1/2.5 is the sampling frequency
    -->
    0/2.5/2, 0.1/2.5/2, ..., 1/2.5/2  # dividing every frequency by 2 so that the max frequency is the Nyquist frequency, which is half the sampling rate (1/2.5/2)
    """
    if fmax == np.inf:
        fMaxIdx = Nf
        fmax = fCoords[-1]
    else:
        fMaxIdx = np.abs(fCoords - fmax).argmin()

    if fmin == 0:
        fMinIdx = 0
    else:
        fMinIdx = np.abs(fCoords - fmin).argmin()

    print(f'fmax: {fmax}')


    if isinstance(dataList, (list, tuple)):
        data = np.sum(dataList, axis = 0)[:ND]
    else:
        data = dataList[:ND]

    dataFFT = np.fft.fft(data * windows.tukey(ND, 0.01))

    WDMStartTime = time.time()
    waveletAmplitudeMatrix = transform_wavelet_freq(dataFFT, Nf, Nt).T
    WDMEndTime = time.time()

    print(f'WDM transform took {WDMEndTime - WDMStartTime:.2f} s')

    # Transform the frequency domain data (taken from the function argument) to the wavelet domain
    #waveletAmplitudeMatrix = transform_wavelet_freq(data, Nf, Nt).T


    # Take abs value. taking the abs value to remove negative values instead of just adding everything by the min value works better for some reason. haven't thought about why this is
    waveletAmplitudeMatrix = np.abs(waveletAmplitudeMatrix)


    # White noise
    rng = np.random.default_rng()
    noise = np.abs(rng.normal(loc = 0, scale = 1, size = waveletAmplitudeMatrix.shape)) * noiseLevel  # 1e-22 for h (actual signal), 1e-24 for TDI (response from LISA)

    waveletAmplitudeMatrix += noise


    # Noise
    if whiten_f is not None:  # I guess this adds noise?
       if whiten_f is not None:
           interp_whiten_PSD = np.interp(fCoords[1:], whiten_f, whiten_PSD)
           waveletAmplitudeMatrix /= np.sqrt(interp_whiten_PSD[:, np.newaxis])
       else:
           warnings.warn('You need to provide whiten_f when providing whiten_PSD; whiten_PSD is currently being ignored.', UserWarning)


    waveletAmplitudeMatrix = waveletAmplitudeMatrix[fMinIdx:fMaxIdx, :]


    # Some form of normalization before cleaning/LogNorm (i think this helps somehow?)
    waveletAmplitudeMatrix /= waveletAmplitudeMatrix.max()


    #print(f'Matrix tail (top rows in pcolormesh) before cleaning: {waveletAmplitudeMatrix[-20:]}')


    # Clean the matrix from bad values by replacing every bad value with the minimum value in the matrix and taking out negative vals. This may be a necessary step because plt.pcolormesh wasn't giving good results when I plotted before cleaning. Though that could just be pcolormesh messing up or something, because the whole graph was just white and white areas are an issue with pcolormesh sometimes. According to Gemini, pcolormesh might plot nan, inf, and -inf vals as white.
    minAmp = waveletAmplitudeMatrix.min()
    waveletAmplitudeMatrixClean = np.nan_to_num(waveletAmplitudeMatrix, nan=minAmp, posinf=minAmp, neginf=minAmp)  # take out NaN, ∞, and -∞ vals
    #print(f'Matrix min & pos before adding min: {waveletAmplitudeMatrixClean.min()}, {np.argmin(waveletAmplitudeMatrixClean)}')
    #print()
    #print(type(waveletAmplitudeMatrixClean), waveletAmplitudeMatrixClean.shape)
    #waveletAmplitudeMatrixClean += np.abs(waveletAmplitudeMatrixClean.min())  # add the abs value of the minimum value to every value, so that the minimum value in the matrix is 0. This makes it so you can always use LogNorm: see where we set vmin.
    #print(f'Matrix min & pos after adding min: {waveletAmplitudeMatrixClean.min()}, {np.argmin(waveletAmplitudeMatrixClean)}')

    #print(f'Matrix tail after cleaning: {waveletAmplitudeMatrixClean[-20:]}')


    # Figure parameters
    if vmin is None:
        vmin = minAmp + 1e-20  # we can't have vmin = 0 because we might want to use LogNorm. So add a very small value to the vmin.
    if vmax is None:
        vmax = waveletAmplitudeMatrixClean.max()
    
    print(f'VMin: {vmin}; VMax: {vmax}')
    
    if logColor is not None:
        norm = colors.LogNorm(vmin = vmin, vmax = vmax, clip = True)  # if you don't set clip = True, it'll plot every row of zeros as white for some reason, even though matplotlib.colors.Colormap's set_under defaults to 'k'. I have no idea what is happening but sure.  # Edit 6/30: now changing it back to False does nothing? i dont know if this is good or bad
    else:
        norm = colors.Normalize(vmin = vmin, vmax = vmax)
    
    print(f'Matrix has any nan or -∞ or ∞ or complex vals: {np.iscomplex(waveletAmplitudeMatrixClean).any() or (waveletAmplitudeMatrixClean == np.inf).any() or (waveletAmplitudeMatrixClean == -np.inf).any() or (waveletAmplitudeMatrixClean == np.nan).any()}')
    
    
    # Plot with plt.pcolormesh
    fig, ax = plt.subplots(figsize=(10, 7))
    pcolormeshStartTime = time.time()
    plt.pcolormesh(tCoords, fCoords[fMinIdx:fMaxIdx + 1], waveletAmplitudeMatrixClean, cmap = cmap, shading = 'auto', snap = False, rasterized = True, norm = norm)
    pcolormeshEndTime = time.time()
    print(f'pcolormesh plotting took {pcolormeshEndTime - pcolormeshStartTime:.2f} s')
    gridLabelScaleLimStartTime = time.time()

    if plotFeatures:
        plt.xlabel('t (s)')
        plt.ylabel('f (Hz)')
        plt.title('WDM Spectrogram')
    else:
        # Courtesy of Gemini
        # 1. Turn off the axes
        ax.set_axis_off()

        # 2. Strip padding from the internal Axes object
        ax.xaxis.set_major_locator(plt.NullLocator())
        ax.yaxis.set_major_locator(plt.NullLocator())
    
    plt.grid(False)
    plt.xlabel('t (s)')
    plt.ylabel('f (Hz)')
    
    
    # Function x**(1/2)
    def forward(x):
        return x**(1/2)
    
    
    def inverse(x):
        return x**2
    
    if logFreq:
        #ax.set_yscale('function', functions = (forward, inverse))
        ax.set_yscale('asinh', linear_width = 0.0005)  # 0.001 is good, 0.0001 is alright, 0.0005 is also good
        #ax.set_yscale('log')
    ax.set_ylim(fmin, fmax)
    gridLabelScaleLimEndTime = time.time()
    print(f'grid, axis labels, scale, and limits took {gridLabelScaleLimEndTime - gridLabelScaleLimStartTime:.2f} s')
    #ax.set_ylim(10**(-4), 10**(-1))
    
    #tMeshgrid, fMeshgrid = np.meshgrid(tCoords, fCoords)
    #ax.scatter(tMeshgrid.flat, fMeshgrid.flat, marker = 'o', color = 'lime', s = 0.001, rasterized = True)
    #ax.axis('tight')
    
    plt.title('pcolormesh')
    plt.tight_layout()
    #savefigStartTime = time.time()
    if filename is not None:
        plt.savefig(filename, dpi = 100)
    #savefigEndTime = time.time()
    #print(f'savefig took {savefigEndTime - savefigStartTime:.2f} s')
    pltShowStartTime = time.time()
    plt.show()
    pltShowEndTime = time.time()
    print(f'plt.show() took {pltShowEndTime - pltShowStartTime:.2f} s')

    #print(f'Number of data points greater than 1.029e-11: {waveletAmplitudeMatrixClean[waveletAmplitudeMatrixClean > 1.0299e-11].size}')
    print(f'ampMatrix shape: {waveletAmplitudeMatrixClean.shape}; len(tCoords): {len(tCoords)}; len(fCoords): {len(fCoords)}')

    return waveletAmplitudeMatrixClean, tCoords, fCoords, fig, ax


In [ ]:
import time
import os
from pathlib import Path
from matplotlib import colors

def getWDMMatrix(waveform, Nt = 128//1, Nf = None, noiseLevel = 1e-24):
    if Nf is None:
        Nf = len(data) // Nt
    ND = Nf * Nt

    dataFFT = np.fft.fft(waveform[:ND])

    waveletAmplitudeMatrix = transform_wavelet_freq(dataFFT, Nf, Nt).T

    # Take abs value. taking the abs value to remove negative values instead of just adding everything by the min value works better for some reason. haven't thought about why this is
    waveletAmplitudeMatrix = np.abs(waveletAmplitudeMatrix)

    # White noise
    if noiseLevel is not None and noiseLevel > 0:
        rng = np.random.default_rng()
        noise = np.abs(rng.normal(loc = 0, scale = 1, size = waveletAmplitudeMatrix.shape)) * noiseLevel  # 1e-22 for h (actual signal), 1e-24 for TDI (response from LISA)

        waveletAmplitudeMatrix += noise

    # Some form of normalization before cleaning/LogNorm (i think this helps somehow?)
    waveletAmplitudeMatrix /= waveletAmplitudeMatrix.max()

    # Clean the matrix from bad values by replacing every bad value with the minimum value in the matrix and taking out negative vals. This may be a necessary step because plt.pcolormesh wasn't giving good results when I plotted before cleaning. Though that could just be pcolormesh messing up or something, because the whole graph was just white and white areas are an issue with pcolormesh sometimes. According to Gemini, pcolormesh might plot nan, inf, and -inf vals as white.
    minAmp = waveletAmplitudeMatrix.min()
    waveletAmplitudeMatrixClean = np.nan_to_num(waveletAmplitudeMatrix, nan=minAmp, posinf=minAmp, neginf=minAmp)  # take out NaN, ∞, and -∞ vals

    return waveletAmplitudeMatrixClean



def getSpectrogram(WDMMatrix, dt, cmap = 'inferno', vmin = None, vmax = None, logColor = True, logFreq = True, fmin = 1e-8, fmax = None, filename = None, imageFolder = None, showPlot = True, plotFeatures = False):
    # Coords for pcolormesh
    Nf, Nt = WDMMatrix.shape
    Nd = Nf * Nt
    TObs = Nd * dt

    tCoords = np.linspace(0, TObs, Nt + 1)
    fCoords = np.arange(0, Nf + 1) / (2 * dt * Nf)


    # Figure parameters
    if vmin is None:
        vmin = WDMMatrix.min() + 1e-20  # we can't have vmin = 0 because we might want to use LogNorm. So add a very small value to the vmin.
    if vmax is None:
        vmax = WDMMatrix.max()

    if logColor:
        norm = colors.LogNorm(vmin = vmin, vmax = vmax, clip = True)  # if you don't set clip = True, it'll plot every row of zeros as white for some reason, even though matplotlib.colors.Colormap's set_under defaults to 'k'. I have no idea what is happening but sure.  # Edit 6/30: now changing it back to False does nothing? i dont know if this is good or bad
    else:
        norm = colors.Normalize(vmin = vmin, vmax = vmax)


    # Plot with plt.pcolormesh
    fig, ax = plt.subplots(figsize=(10, 7))

    plt.pcolormesh(tCoords, fCoords, WDMMatrix, cmap = cmap, shading = 'auto', snap = False, rasterized = True, norm = norm)

    plt.grid(False)

    if plotFeatures:
        plt.xlabel('t (s)')
        plt.ylabel('f (Hz)')
        plt.title('WDM Spectrogram')
    else:
        # Courtesy of Gemini
        # 1. Turn off the axes
        ax.set_axis_off()

        # 2. Strip padding from the internal Axes object
        ax.xaxis.set_major_locator(plt.NullLocator())
        ax.yaxis.set_major_locator(plt.NullLocator())


    if logFreq:
        ax.set_yscale('asinh', linear_width = 0.0005)  # 0.001 is good, 0.0001 is alright, 0.0005 is also good

    ax.set_ylim(fmin, fmax)
    plt.tight_layout()

    if showPlot:
        plt.show()

    if filename is not None:
        plt.savefig(Path(imageFolder) / (filename + '.jpg'), dpi = 300, pad_inches = 0.0)

    # If you don't do this, Jupyter Notebook will show you the plot either way (which ig is nice bc you don't need to always do plt.show() for it to show?)
    plt.close()


    return fig, ax, tCoords, fCoords




window function (specifically Tukey, seemingly at any alpha value) seems to fix the decreasing brightness issue

Friday Jun. 12 notes:
- we need to figure out if there being no data above the merger is okay. it seems that whenever a given row has some sort of amplitude from the merger, there's other instrument noise in the row and there's never a nonzero value (I think, because pcolormesh seems to have been plotting 0 values as white, but I haven't actually fully checked the data), but when there isn't any amplitude (BHs coalesced) the rest of the row is just 0? unless those rows are being 0'd out some other way, in which case obviously both the merger and instrument noise/small-signal data would be 0 anyway
- we need to figure out how to get rid of the horizontal line that extends to the right for some reason sometimes at the lower frequencies of a given merger. maybe we can use some sort of window after the WDM transform? idk how that would work really, because we kind of need to dynamically know which frequency the horizontal line will be at, right? unless we can steal however BBHx actually inserts the BHB waveform, then we could figure it out, but without that I think it would be very difficult
- maybe use scipy spectrogram instead of the WDM transform? at least try it
- why is it that in Robbie's original spectrogram, we don't see the horizontal lines for BHBs really, like is the signal just that dim when they get horizontal?
- figure out why he forces Nf and Nt to be even
- figure out why Nf is the number of data points per wavelet
- figure out how to make the time and frequency axes accurate in the image (currently, the amplitude data matrix doesn't exactly line up with at least the frequencies (and maybe the times?)). BBHx's FD waveform lets us specify a frequency array, but we're currently using Nf as a value not equal to the length of that frequency array (instead, ND = the length of that array, and ND (= len(freqs)) = Nf * Nt, and we're saying Nf = WAVELET_DURATION / dt with dt being (t_obs_end - t_obs_start) / ND, and I'm not sure if these are correct or not. Maybe we can ask Robbie?
- bbhx doesn't work when direct = True?


Monday Jun. 15:
- why does Robbie do an FFT of the time series data instead of just using transform_wavelet_time? i tried both and it seemingly has no difference when using phentax, though with BBHx (or maybe just with that horizontal line) using transform_wavelet_time gives a vertical line.. haven't tested it with the original simulation data, so maybe there was some sort of artifact. if not, it just seems like an extra step for no reason to me

### Phentax

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(["science", "notebook"])

from phentax.waveform import IMRPhenomTHM

from lisaconstants import ASTRONOMICAL_YEAR
from lisatools.detector import EqualArmlengthOrbits
from fastlisaresponse import ResponseWrapper  # also can do from fastlisaresponse.response import ResponseWrapper; ResponseWrapper is located in fastlisaresponse.response


def instantiateResponseFunc(dt, Tobs, allModes = False):
    # Step 1: Instantiate the IMRPhenomTHM object from phentax ---------------------------------------------------------------
    
    tlowfit = True # use a fit to set the starting time of the root finder used in t(f)
    tol = 1e-12 # root finding tolerance
    higher_modes = "all" if allModes else None
    
    imr = IMRPhenomTHM(
            higher_modes=higher_modes,  # higher_modes="all" for all modes
            include_negative_modes=True, # negative m modes will be produced by symmetry
            t_low_fit=tlowfit,
            coarse_grain=False, # if false it will generate the waveform on a dense time grid with the specified timestep
            atol=tol,
            rtol=tol,
            T=Tobs,
        )
    
    
    # Step 2: Initialize the ResponseWrapper object (it's basically a function, main attribute is __call__) ------------------
    
    # Some fastlisaresponse ResponseWrapper params
    index_lambda, index_beta = 0, 1
    force_backend = "cpu"
    orbits = EqualArmlengthOrbits()
    
    def compute_polarizations_at_once_new(m1, m2, chi1, chi2, distance, phi_ref, inclination, psi, dt, f_min, f_ref, T):
        times, mask, hplus, hcross = imr.compute_polarizations_at_once(m1 = m1, m2 = m2, chi1z = chi1, chi2z = chi2, distance = distance, phi_ref = phi_ref, inclination = inclination, psi = psi, f_min = f_min, f_ref = f_ref, delta_t = dt)
        return hplus[mask] + 1j * hcross[mask]

    lisaResponseFunc = ResponseWrapper(
        waveform_gen = compute_polarizations_at_once_new,
        Tobs = Tobs / ASTRONOMICAL_YEAR,  # Tobs is in years (can't we just always use SI?)
        dt = dt,
        index_lambda = index_lambda,
        index_beta = index_beta,
        flip_hx = False,
        remove_sky_coords = True,
        remove_garbage = True,
        force_backend = force_backend,
        orbits = orbits
        )


    # Step 3: Return the response function -----------------------------------------------------------------------------------

    return lisaResponseFunc


def getResponse(logmT, q, chi1, chi2, distance, cosinc, phi_ref, psi, f_min, f_ref, sinbeta, lam, dt, Tobs, allModes = False, responseFunc = None):
    # Step 1: Instantiate the response function as needed --------------------------------------------------------------------

    if responseFunc is None:
        responseFunc = instantiateResponseFunc(dt, Tobs, allModes = allModes)


    # Step 2: Get the response -----------------------------------------------------------------------------------------------

    mT = 10**logmT
    m1 = mT/(q + 1)
    m2 = mT - m1

    inclination = np.arccos(cosinc)
    beta = np.arcsin(sinbeta)

    waveA, waveE, waveT = responseFunc(
        beta,
        lam,
        m1 = m1,
        m2 = m2,
        chi1 = chi1,
        chi2 = chi2,
        distance = distance,
        phi_ref = phi_ref,
        inclination = inclination,
        psi = psi,
        dt = dt,
        f_min = f_min,
        f_ref = f_ref
    )
    
    
    # Step 3: Return the waveforms -------------------------------------------------------------------------------------------
    return waveA, waveE, waveT



In [ ]:
tlowfit = True # use a fit to set the starting time of the root finder used in t(f)
tol = 1e-12 # root finding tolerance
Tobs = 5 * ASTRONOMICAL_YEAR / 12
dt = 2.5

imr = IMRPhenomTHM(
        higher_modes=None,  # higher_modes="all" for all modes
        include_negative_modes=True, # negative m modes will be produced by symmetry
        t_low_fit=tlowfit,
        coarse_grain=False, # if false it will generate the waveform on a dense time grid with the specified timestep
        atol=tol,
        rtol=tol,
        T=Tobs,
    )

In [ ]:
from lisatools.detector import EqualArmlengthOrbits

# Some fastlisaresponse ResponseWrapper params
index_lambda, index_beta = 0, 1
force_backend = "cpu"
orbits = EqualArmlengthOrbits()

In [ ]:
def compute_polarizations_at_once_new(m1, m2, chi1, chi2, distance, phi_ref, inclination, psi, dt, f_min, f_ref, T):
    times, mask, hplus, hcross = imr.compute_polarizations_at_once(m1 = m1, m2 = m2, chi1z = chi1, chi2z = chi2, distance = distance, phi_ref = phi_ref, inclination = inclination, psi = psi, f_min = f_min, f_ref = f_ref, delta_t = dt)
    return hplus[mask] + 1j * hcross[mask]


In [ ]:
lisaResponseFunc = ResponseWrapper(
        waveform_gen = compute_polarizations_at_once_new,
        Tobs = Tobs / ASTRONOMICAL_YEAR,  # Tobs is in years (can't we just always use SI?)
        dt = dt,
        index_lambda = index_lambda,
        index_beta = index_beta,
        flip_hx = False,
        remove_sky_coords = True,
        remove_garbage = True,
        force_backend = force_backend,
        orbits = orbits
        )

In [ ]:
m1 = 1e5
m2 = 6e4
chi1 = 0.9
chi2 = 0.3
distance = 1e4
inclination = np.pi / 3.0
phi_ref = 0.0
psi = 1.0
f_min = 1e-8
f_ref = f_min
# t_ref = 0.0
beta = 0.9805742971871619
lam = 5.22979888

# m1s = jnp.array([1e6, 1e3, 5e7])
# m2s = jnp.array([3e5, 1e3, 9e6])
# chi1s = jnp.array([0.9, 0.6, 0.7])
# chi2s = jnp.array([0.3, 0.1, 0.5])
# distances = jnp.array([1e3, 500, 2e4])
# inclinations = jnp.array([jnp.pi / 3.0, jnp.pi / 6.0, jnp. pi / 9.0])
# phi_refs = jnp.array([0.0] * 3)
# psis = jnp.array([1.0, 0.7, 1.3])
# f_mins = jnp.array([1e-6, 1e-6, 1e-6])
# f_refs = f_mins
# betas = jnp.array([0.97, 0.4, 1.3])
# lams = jnp.array([5.2, 0.1, np.pi])
# dts = jnp.array([dt] * 3)

In [ ]:
from num2words import num2words
times, mask, h_plus, h_cross = imr.compute_polarizations_at_once(m1 = m1s, m2 = m2s, chi1z = chi1s, chi2z = chi2s, distance = distances, phi_ref = phi_refs, inclination = inclinations, psi = psis, f_min = f_mins, f_ref = f_refs, delta_t = dt)

#print(h_plus.shape)
#print(mask.shape)
#print(h_plus[mask].shape)

#print(len([binary[mask[idx]] for idx, binary in enumerate(h_plus)][0]))
#print(h_plus)

print(times.shape)

for k in range(len(times)):
    plt.figure()
    #ordinal = num2words(k + 1, to = 'ordinal')
    #ordinal = ordinal[0].capitalize() +
    plt.title(f'{num2words(k + 1, to = 'ordinal').capitalize()} Binary')
    #plt.plot(times[k][mask[k]], h_plus[k][mask[k]], color = 'blue', label = 'masked', rasterized = True, zorder = 2)
    plt.plot(times[k], h_plus[k], color = 'red', label = 'unmasked', rasterized = True)
    plt.legend()
    #plt.xlim(-0.1e6,0.1e6)


In [ ]:
waveA, waveE, waveT = lisaResponseFunc(
        beta,
        lam,
        m1 = m1,
        m2 = m2,
        chi1 = chi1,
        chi2 = chi2,
        distance = distance,
        phi_ref = phi_ref,
        inclination = inclination,
        psi = psi,
        dt=dt,
        f_min=f_min,
        f_ref=f_ref
    )

In [ ]:
print(len(waveA))
plt.plot(np.sqrt(waveA**2) / np.max(np.sqrt(waveA**2)))
plt.axhline(1e-2)
plt.semilogy()
plt.title('TDI A')
#plt.savefig('TDI Example.jpg')

#plt.figure()
#plt.plot(h_plus[mask])
#plt.title('h_plus')
#plt.savefig('h_plus example.jpg')

In [ ]:
print(len(waveA))
plt.plot(np.log10(np.abs(waveA)))
plt.axhline(-23)
#plt.semilogy()
plt.title('TDI A')
#plt.savefig('TDI Example.jpg')

#plt.figure()
#plt.plot(h_plus[mask])
#plt.title('h_plus')
#plt.savefig('h_plus example.jpg')

In [ ]:
waveAs, waveEs, waveTs = [], [], []
for idx in range(len(m1s)):
    waveA_idx, waveE_idx, waveT_idx = lisaResponseFunc(
        betas[idx],
        lams[idx],
        m1 = m1s[idx],
        m2 = m2s[idx],
        chi1 = chi1s[idx],
        chi2 = chi2s[idx],
        distance = distances[idx],
        phi_ref = phi_refs[idx],
        inclination = inclinations[idx],
        psi = psis[idx],
        dt=dt,
        f_min=f_mins[idx],
        f_ref=f_refs[idx]
    )
    waveAs.append(waveA_idx)
    waveEs.append(waveE_idx)
    waveTs.append(waveT_idx)

In [ ]:
plt.plot(waveAs[0])
plt.title('First Binary')

plt.figure()
plt.plot(waveAs[1])
plt.title('Second Binary')

plt.figure()
plt.plot(waveAs[2])
plt.title('Third Binary')


In [ ]:
def placeXIntoY(x, y, xIdx = 0, yIdx = 0, add = True, windowFunc = None, windowFuncArgs = [], fadeLengthStart = 0):  # TODO: add functionality for not specifying xIdx and/or yIdx, handle add vs. set, write docstring. i thought of adding a robust window function capability where you can pass in a window function (say scipy.signal.windows.tukey) and arguments and it'll apply the function, but if you want it windowed you can just pass x pre-windowed. fun learning idea though...
    """
    Places an array x into another array y at a given location yIdx that corresponds to xIdx.
    """

    # Old code ---------------------------------------------
    # # TODO: maybe rework the logic so that both of the 1st 2 cases are handled simultaneously (i.e. the first and last indices of totTimeSeries are determined, and the truncation of the original time series is determined, and then the addition is done). we could have variables like totTimeSeriesStartIdx and timeSeriesStartIdx, etc. to handle truncation and placement dynamically. Though, that might make it a bit more difficult to understand? idk. also we can make this into a function
    # if (tCIdx < zeroIdx):  # Handle cases where the coalescence time is too close to t = 0 and we need to truncate the first bit of the time series data.
    #     tIdxDiff = zeroIdx - tCIdx
    #     totTimeSeries[:tCIdx + (numMaskedTimes - zeroIdx)] += totalSignalMasked[tIdxDiff:] * windows.tukey(len(totalSignalMasked[tIdxDiff:]), 0.1)  # for the first one, can do numMaskedTime - tIdxDiff or tCIdx + (numMaskedTimes - zeroIdx).
    #     """
    #     Concrete example of how this works:
    #     Say my time series indices looks like this: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    #     And let's say zeroIdx = 6 (that's where the coalescence happens)
    #     And let's say we need index 4 in totTimeSeries to be the coalescence time (tCIdx)
    #     Then tIdxDiff will be 6 - 4 = 2, and the next line will add to the first 8 (= tCIdx + (numMaskedTimes - zeroIdx) = 4 + 10 - 6) values in totTimesSeries by totalSignalMasked[2:] (which gives indices 2 to 9, which is 8 values).
    #     Let's see if index 4 in the new array (totTimeSeries) is the coalescence time (index 6 in the original array): 2, 3, 4, 5, 6 --> 0, 1, 2, 3, 4, so yes, the original index 6 corresponds to index 4 in the new array!
    #     Slap on a window function and we're good to go!
    #     Note that this logic doesn't take into account the edge case in which tCIdx is less than zeroIdx but the number of points after zeroIdx in the time series is greater than the number of points in totTimeSeries after tCIdx, but that won't happen since the total observation time will be long enough and there aren't that many points in the original time series after the coalescence occurs.
    #     """
    # elif (numTotTimes - tCIdx < numMaskedTimes - zeroIdx):  # Handle cases where the coalescence time is too close to t = TObsTot (2 yrs in our case) and we need to truncate the last bit of the time series data. Also, I don't like "elif" >:(
    #     tIdxDiff = (numMaskedTimes - zeroIdx) - (numTotTimes - tCIdx)
    #     totTimeSeries[tCIdx - zeroIdx:] += totalSignalMasked[:numMaskedTimes - tIdxDiff] * windows.tukey(len(totalSignalMasked[:numMaskedTimes - tIdxDiff]), 0.1)
    # else:  # If we're here, that means the coalescence time is nicely in the middle of the observation window and not too close to either of the edges.
    #     totTimeSeries[tCIdx - zeroIdx: tCIdx - zeroIdx + numMaskedTimes] += totalSignalMasked * windows.tukey(numMaskedTimes)
    # End of old code --------------------------------------

    def halfCosWindow(length):
        """Generates a half-cosine window of a given length."""
        return 0.5 * (1 - np.cos(np.linspace(0, np.pi, length)))


    xLen = len(x)
    yLen = len(y)

    idxDiffStart = yIdx - xIdx
    idxDiffEnd = (yLen - yIdx) - (xLen - xIdx)

    yStart = 0 if idxDiffStart < 0 else idxDiffStart
    yEnd = yLen if idxDiffEnd < 0 else idxDiffStart + xLen
    xStart = -idxDiffStart if idxDiffStart < 0 else 0
    xEnd = xLen + idxDiffEnd if idxDiffEnd < 0 else xLen

    #windowFunction = windowFunc(len(x[xStart:xEnd]), *windowFuncArgs) if windowFunc else 1

    # Windowing the start of the merger and the part after the merger
    xFaded = np.copy(x[xStart:xEnd])
    xFadedLen = len(xFaded)

    # Fade the start of the waveform
    newFadeLengthStart = min(fadeLengthStart, xFadedLen)
    fullWindow = halfCosWindow(fadeLengthStart)
    xFaded[:newFadeLengthStart] *= fullWindow[:newFadeLengthStart]

    # Fade the end of the waveform, from the coalescence up to the point at which the previous 10 points in time were less than 1e-25 in magnitude
    # valSum = 0
    # signalStopIdx = -1
    # if idxDiffStart >= 0:
    #     xIdxNew = xIdx
    # else:
    #     xIdxNew = xIdx + idxDiffStart
    # for idx, val in enumerate(xFaded[xIdxNew + 1:]):
    #     valSum += np.abs(val)
    #     if idx >= 10:
    #         valSum -= np.abs(xFaded[xIdxNew + 1:][idx - 10])
    #         if valSum / 10 <= 1e-25:
    #             signalStopIdx = idx
    #             break
    # if signalStopIdx != -1:
    #     xFaded[xIdxNew + 1:(xIdxNew + 1) + (signalStopIdx + 1)] *= halfCosWindow(signalStopIdx + 1)[::-1]
    # print(f'signalStopIdx: {signalStopIdx}')

    #print((xFaded == x[xStart:xEnd])[:200])

    # Fade the end of the waveform
    xIdxNew = xIdx if idxDiffStart >= 0 else xIdx + idxDiffStart
    xFaded[xIdxNew + 1:] *= halfCosWindow(xFadedLen - (xIdxNew + 1))[::-1]


    if add:
        y[yStart:yEnd] += xFaded
    else:
        y[yStart:yEnd] = xFaded

    return y



 ### Plotting h_plus, unpadded and padded

In [ ]:
from scipy.ndimage import gaussian_filter1d

print(len(mask[0]), len(times[0]), len(h_plus[0]), len(h_cross[0]))

fig = plt.figure(figsize=FIGSIZE)
timesMasked, hPlusMasked, hCrossMasked = times[mask], h_plus[mask], h_cross[mask]
#timesMasked, hPlusMasked, hCrossMasked = times[mask], h_plus[mask], h_cross[mask]
totalSignalMasked = hPlusMasked + hCrossMasked * 1j  # h_+ + ih_x = signal
totalSignalMasked = hPlusMasked
plt.plot(timesMasked, hPlusMasked, rasterized = True)
#plt.plot(times, h_cross, rasterized = True)
#plt.legend()
plt.xlabel("Time (s)")
plt.ylabel("Strain")
plt.title('Time Domain Waveform (unpadded)')
plt.xlim(0, 1250)
plt.tight_layout()
plt.show()


#times = np.linspace()

tC = 1.4 * ASTRONOMICAL_YEAR  # desired coalescence time starting from t = 0 in the whole observation
tCIdx = int(tC / dt)

TObsTot = 2.0 * ASTRONOMICAL_YEAR
numTotTimes = int(TObsTot / dt)  # this is the total number of time data points we will always have; if the number of data points is less than this, we will pad the data with zeros (using a window function) to make sure the total time is always this number. we end up getting a time that's slightly under our desired time in most cases (that's at the very least true for TObsTot = 2 yrs).
totTimeSeries = np.zeros(numTotTimes)

numMaskedTimes = len(timesMasked)

zeroIdx = np.argmin(np.abs(timesMasked))  # estimated index, within the time series array, of the coalescence (t = 0 in the time series)



fadeLengthStart = len(totalSignalMasked) // 20


placeXIntoY(totalSignalMasked, totTimeSeries, zeroIdx, tCIdx, windowFunc = windows.tukey, windowFuncArgs = [0.01], fadeLengthStart = fadeLengthStart)

#nPrintTDWF = 100
#print(f'Last {nPrintTDWF} of waveform: {totalSignalMasked[-nPrintTDWF:]}')

totTimeSeriesTimes = np.arange(0, numTotTimes * dt, dt)  # arange doesn't include the stop value


plt.figure()
plt.plot(totTimeSeriesTimes, totTimeSeries)
plt.xlabel('Time (s)')
plt.ylabel('Strain')
#plt.xlim((1.39999) * ASTRONOMICAL_YEAR, (1.4001) * ASTRONOMICAL_YEAR)
plt.title(f'Time Domain Waveform (padded to {numTotTimes * dt / ASTRONOMICAL_YEAR:.2f} yrs), coalescence time {tC / ASTRONOMICAL_YEAR:.2f} yrs')




### Plotting TDI A, unpadded and padded

In [ ]:
def TDIPlacement(totTimeSeries, tC, TDIData, dt, plot = True):
    """
    totTimeSeries: total time series that you want to place the binary waveform in
    tC: coalescence time in total observation time array (s), starting from t = 0 in the whole observation array
    TDIData: TDI time-domain waveform data (amplitude list)
    dt: amount of time between observations (s)
    plot: whether or not to plot the unpadded and padded (placed) waveforms; True by default
    """

    # Unpadded Time Series ---------------------------------------------------
    dataLen = len(TDIData)

    # Plot
    if plot:
        # Make time array
        dataTimes = np.arange(0, dataLen * dt, dt)

        plt.figure(figsize=FIGSIZE)
        plt.plot(dataTimes, TDIData)
        plt.xlabel('Time (s)')
        plt.ylabel('Strain')
        plt.title('Time Domain Waveform (unpadded)')
        plt.tight_layout()


    # Padded Time Series -----------------------------------------------------
    # Get coalescence time index
    tCIdx = int(tC / dt)

    # Get the coalescence time from the TDI data
    tCData = np.argmax(np.abs(TDIData))


    # Place waveA into the total time series
    fadeLengthStart = dataLen // 20
    placeXIntoY(TDIData, totTimeSeries, tCData, tCIdx, fadeLengthStart = fadeLengthStart)

    # Plot
    if plot:
        # Get total time series time array
        totTimeSeriesTimes = np.arange(0, numTotTimes * dt, dt)

        plt.figure(figsize = FIGSIZE)
        plt.plot(totTimeSeriesTimes, totTimeSeries)
        plt.xlabel('Time (s)')
        plt.ylabel('Strain')
        plt.title('Time Domain Waveform (padded)')
        plt.tight_layout()

In [ ]:
TObsTot = 2.0 * ASTRONOMICAL_YEAR
numTotTimes = int(TObsTot / dt)  # this is the total number of time data points we will always have; if the number of data points is less than this, we will pad the data with zeros (using a window function) to make sure the total time is always this number. we end up getting a time that's slightly under our desired time in most cases (that's at the very least true for TObsTot = 2 yrs).
totTimeSeries = np.zeros(numTotTimes)

tCs = np.array([1.4, 0.6, 1.8]) * ASTRONOMICAL_YEAR

#for k in range(len(tCs)):
#    TDIPlacement(totTimeSeries, tCs[k], waveAs[k], dt)

tC = 0.8 * ASTRONOMICAL_YEAR
TDIPlacement(totTimeSeries, tC, waveA, dt)



In [ ]:
waveAFFT = np.fft.fft(waveA)
waveAFFTFreqs = np.fft.fftfreq(len(waveA), d = dt)

plt.plot(waveAFFTFreqs, np.abs(waveAFFT))
plt.loglog()
plt.xlim(0, 0.2)


waveA1FFT = np.fft.fft(waveAs[1])
waveA1FFTFreqs = np.fft.fftfreq(len(waveAs[1]), d = dt)

plt.figure()
plt.plot(waveA1FFTFreqs, np.abs(waveA1FFT))

In [ ]:
plt.plot(windows.tukey(1000, 0.3))
print(np.argmin(np.abs(windows.tukey(1000, 0.3) - 1)))
plt.scatter(150, 1, color = 'orange', zorder = 2)


def halfCosWindow(length):
    """Generates a half-cosine window of a given length."""
    return 0.5 * (1 - np.cos(np.linspace(0, np.pi, length)))

frontWindowLength = 100

# 1. Generate the window
window_length = 100
window = halfCosWindow(window_length)

# 2. Visualize the window
plt.figure(figsize=(6, 6))
plt.plot(window, color='b', linewidth=2)
plt.title("Half-Cosine Window")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.grid(True, linestyle='--')
plt.show()



def apply_half_cosine_fade(signal, fade_length=1000):
    """Applies a half-cosine fade-in and fade-out to a signal."""
    # Create the half-cosine envelope
    fade_envelope = halfCosWindow(fade_length)

    # Create the tapered signal copy
    tapered_signal = signal.copy()

    # Apply fade-in to the start
    tapered_signal[:fade_length] *= fade_envelope

    # Apply fade-out to the end
    tapered_signal[-fade_length:] *= fade_envelope[::-1]

    return tapered_signal

# Example with a dummy 1-second audio signal (e.g., 44100 samples)
sample_rate = 44100
t = np.linspace(0, 1, sample_rate, endpoint=False)
raw_audio = np.sin(2 * np.pi * 440 * t)  # 440 Hz sine wave

# Apply fade
faded_audio = apply_half_cosine_fade(raw_audio, fade_length=2000)



In [ ]:
WAVELET_DURATION = 3600 * 3

NT4 = int(TObsTot / WAVELET_DURATION)
ND4 = len(totTimeSeries)
NF4 = int(ND4 / NT4)

# Force NF4 and NT4 to be even because WDMWaveletTransforms says it assumes they're even and if they're not then results can be inaccurate.
if NF4 % 2 != 0:
    NF4 -= 1
if NT4 % 2 != 0:
    NT4 -= 1

ND4 = NF4 * NT4

print(f'ND4: {ND4}; NT4: {NT4}; NF4: {NF4}')

In [ ]:
%%time

spectrogramMatrix, tCoords, fCoords, fig1, ax1 = waveletSpectrogram(totTimeSeries[:ND4], dt, Nt=NT4, Nf=NF4, filename="sgramTest.jpg", vmin = 1e-4, noiseLevel = 10**(-22.5))#, cmap = 'Dark2')


In [ ]:
"""
My "old" code


def getBoundingBox(matrix, tCIdx, threshold, tObsIdxSize, fMinIdx, lineExtension = -1):
    '''
    Get a bounding box for a merger.

    matrix (array-like): A 2d matrix containing a spectrogram with a single BHB.
    tCIdx (int): the index of the coalescence time. This is also the right boundary of the box.
    threshold: the brightness threshold for determining the other 3 boundaries
    lineExtension (int): the width of the line (in pixels) added to each side of the center of the line, used for finding the other 3 boundaries.
    tObsIdxSize (int): TObs, converted to pixels. Used to determine the starting column for the left boundary.
    '''

    if not isinstance(matrix, np.ndarray):
        matrix = np.array(matrix)

    rightBoundary = tCIdx
    nRows, nCols = matrix.shape

    leftExtend = lineExtension if tCIdx >= lineExtension else tCIdx
    rightExtend = lineExtension if (nCols - 1) - tCIdx >= lineExtension else (nCols - 1) - tCIdx
    leftStart = tCIdx - tObsIdxSize if tCIdx >= tObsIdxSize else 0

    topBoundary = bottomBoundary = leftBoundary = -1

    # Find top boundary
    for row in range(nRows)[::-1]:
        vals = matrix[row, tCIdx - leftExtend:tCIdx + rightExtend + 1] if lineExtension != -1 else matrix[row, :]
        if (vals >= threshold).any():
            topBoundary = row
            break

    # Find left and bottom boundaries
    for col in range(leftStart, tCIdx):  # go up to tCIdx - 1 since tCIdx is the right boundary
        evalArray = matrix[:fMinIdx - 1:-1, col] >= threshold
        bottomBoundaryGuess = evalArray.argmax()
        if not evalArray[bottomBoundaryGuess]:
            continue

        leftBoundary, bottomBoundary = col, bottomBoundaryGuess
        break

    warningsTextArr = np.array(["Top", "Bottom", "Left"])
    boundariesArr = np.array([topBoundary, bottomBoundary, leftBoundary])
    if (boundariesArr == -1).any():
        warnings.warn(f"The following boundaries were not found: {", ".join(warningsTextArr[boundariesArr == -1])}", UserWarning, stacklevel = 2)

    return leftBoundary, rightBoundary, bottomBoundary, topBoundary


"""

In [ ]:
import numpy as np
import warnings

def getBoundingBox(matrix, tCIdx, leftThreshold, tObsIdxSize, fMinIdx, lineExtension = -1, manualTopBoundaryY = None, topThreshold = None, logNormalize = True):
    """
    Get a bounding box for a merger. (Made by me and slightly fixed by Gemini.)

    matrix (array-like): A 2d matrix containing a spectrogram with a single BHB.
    tCIdx (int): the index of the coalescence time. This is also the right boundary of the box.
    threshold: the brightness threshold for determining the other 3 boundaries
    lineExtension (int): the width of the line (in pixels) added to each side of the center of the line, used for finding the other 3 boundaries.
    tObsIdxSize (int): TObs, converted to pixels. Used to determine the starting column for the left boundary.
    manualTopBoundaryY (int, optional): manually set the top boundary index.
    """

    if not isinstance(matrix, np.ndarray):
        matrix = np.array(matrix)

    newMatrix = matrix.copy()

    if logNormalize:
        # vmin = newMatrix.min()
        # if vmin <= 0:
        #     vmin = 1e-20
        vmin = 1e-4
        vmax = newMatrix.max()

        logVmin = np.log10(vmin)
        logVmax = np.log10(vmax)
        logData = np.log10(newMatrix)

        newMatrix = (logData - logVmin) / (logVmax - logVmin)
        newMatrix = np.clip(newMatrix, 0, 1)


    rightBoundary = tCIdx
    nRows, nCols = newMatrix.shape

    zeroIdx = int(False)
    posOne = int(True)
    negOne = -1

    leftExtend = lineExtension if tCIdx >= lineExtension else tCIdx
    rightExtend = lineExtension if (nCols - 1) - tCIdx >= lineExtension else (nCols - 1) - tCIdx
    leftStart = tCIdx - tObsIdxSize if tCIdx >= tObsIdxSize else zeroIdx

    topBoundary = bottomBoundary = leftBoundary = negOne

    # find top boundary
    if manualTopBoundaryY is not None:
        topBoundary = manualTopBoundaryY if manualTopBoundaryY < nRows else nRows - 1
    else:
        for row in reversed(range(nRows)):
            if lineExtension != negOne:
                leftSlice = tCIdx - leftExtend
                rightSlice = tCIdx + rightExtend + posOne
                vals = newMatrix[row, leftSlice:rightSlice]
            else:
                vals = newMatrix[row, :]

            if (vals >= topThreshold).any():
                topBoundary = row
                break

    # find left and bottom boundaries
    for col in range(leftStart, tCIdx):
        colData = newMatrix[fMinIdx:, col]
        validPixels = np.where(colData >= leftThreshold)[zeroIdx]

        if validPixels.size > zeroIdx:
            leftBoundary = col
            # grab the topmost pixel of the signal in this specific column
            bottomBoundary = fMinIdx + validPixels.max()
            break

    warningsTextArr = np.array(["Top", "Bottom", "Left"])
    boundariesArr = np.array([topBoundary, bottomBoundary, leftBoundary])

    if (boundariesArr == negOne).any():
        missingMask = boundariesArr == negOne
        missingBoundaries = ", ".join(warningsTextArr[missingMask])
        warnings.warn(f"The following boundaries were not found: {missingBoundaries}", UserWarning, stacklevel = 2)

    return leftBoundary, rightBoundary, bottomBoundary, topBoundary

In [ ]:
newMatrix = spectrogramMatrix.copy()

vmin = 1e-4
vmax = np.max(newMatrix)

logVmin = np.log10(vmin)
logVmax = np.log10(vmax)
logData = np.log10(newMatrix)

normSpectrogramMatrix = (logData - logVmin) / (logVmax - logVmin)

normSpectrogramMatrix = np.clip(normSpectrogramMatrix, 0, 1)


fCoordSpacing = 1 / WAVELET_DURATION / 2  # 1 / WAVELET_DURATION for frequency of wavelet; / 2 for Nyquist. Can also do fCoords[-1] / len(fCoords).
fISCO = 4400 / (m1 + m2) * 1.3
print(fISCO)

# TODO: round() in the tdiplacement and stuff instead of int()?
leftBoundary, rightBoundary, bottomBoundary, topBoundary = getBoundingBox(
    normSpectrogramMatrix,
    round(tC / WAVELET_DURATION),
    threshold = 0.5,  # 0.5247
    tObsIdxSize = int(Tobs / WAVELET_DURATION),
    fMinIdx = 1,
    manualTopBoundaryY = round(fISCO / fCoordSpacing),
    logNormalize = False
    )

leftBoundary, rightBoundary, bottomBoundary, topBoundary

In [ ]:
fCoordSpacing = 1 / WAVELET_DURATION / 2  # 1 / WAVELET_DURATION for frequency of wavelet; / 2 for Nyquist. Can also do fCoords[-1] / len(fCoords).

#print(leftRightBound * WAVELET_DURATION, bottomBoundary * fCoordSpacing, topBoundary * fCoordSpacing)
#print(round(bottomTopBound * fCoordSpacing), round(leftBoundary * WAVELET_DURATION), round(rightBoundary * WAVELET_DURATION))

for leftRightBound in [leftBoundary, rightBoundary]:
   ax1.vlines(leftRightBound * WAVELET_DURATION, bottomBoundary * fCoordSpacing, topBoundary * fCoordSpacing, lw = 0.3, color = 'lime')
for bottomTopBound in [bottomBoundary, topBoundary]:
   ax1.hlines(bottomTopBound * fCoordSpacing, leftBoundary * WAVELET_DURATION, rightBoundary * WAVELET_DURATION, lw = 0.3, color = 'lime')

fig1

Tue. Jun 16:
- priors are the distributions of parameters you're choosing for your simulation (or in our case, NN training) data. so choose certain params from distributions (priors) (masses (linear from log dist.), angles (linear from cos(theta) dist), distance (linear from power law bc as distance increases comoving volume increases by distance^3)), maybe use priors from cds1 data
- was getting weird results with what definitely looked like aliasing once i plotted some different BHBs correctly with all the padding. somehow, doing an FFT and then doing the WDM transform from frequency space removes much of the aliasing, though it looks like it might still be somewhat there in the areas very close to the strong BHB signal in the spectrogram? before, it was in a huge vertical band all throughout the BHB times, and now it's only in areas close to the main signal.
- there are vertical lines where i'm inserting the time domain waveform into the array of 0s, and they get reduced when using windowing. however, the effect on the left side is significantly greater than the effect on the right side (like, near the merging event), so we'll have to use an asymmetric windowing function. we may have to dynamically calculate the windowing range to make sure the merging event isn't lessened. also, changing the vmin can help reduce the lines.
- for the bounding boxes, getting tmin and tmax might be pretty easy because we know the indices in the totTimeSeries array (although, it might not be exactly the same as what appears in the spectrogram bc of the WDM transform, like the wavelet duration will determine the time-axis precision... maybe we could calculate what the first wavelet is in the totTimeSeries, like for example, the first 2 hours has nothing, the next 2 hours has nothing, etc... and then the first wavelet that has data is our bounding box start? and then we know the coalescence time index so we can do the same for that? idk... and then for the fmin and fmax, we could maybe do an FFT and get the fmin and fmax there, or since that might not be accurate in the spectrogram, we can calculate the frequency step (size per pixel, which I think for the time is the wavelet duration or the wavelet duration * 2 or something like that) and see which step the first data point is in the FFT? i have no idea honestly

Wed. Jun 17:
- WDMWaveletTransforms (GitHub readme) says Nt and Nf need to be even otherwise results may be inaccurate. It also says the frequency wavelet transform is much faster and more accurate. These certainly answer some questions.

Tue. Jun 23:
- I solved the fastlisaresponse issue on Fri
- Gemini just solved the phentax issue with too high mass ratio
- now we basically need to figure out how to draw bounding boxes
    - t_max is easy, it should just be the coalescence time?
    - t_

In [ ]:
import numpy as np
import pyqtgraph as pg
import pyqtgraph.exporters
from pyqtgraph.Qt import QtCore

# --- Custom Axis for asinh ---
class AsinhAxisItem(pg.AxisItem):
    def __init__(self, linear_width=0.0005, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.linear_width = linear_width

    def tickStrings(self, values, scale, spacing):
        # Reverse the arcsinh transform to display true linear values on the ticks
        strings = []
        for v in values:
            actual_val = np.sinh(v) * self.linear_width
            strings.append(f"{actual_val:.2e}")
        return strings

# 1. Initialize App
app = pg.mkQApp("PColorMesh App")

# 2. Data Preparation
Nx, Ny = 10, 10
linear_width = 0.0005

# Generate linear edges
x_edges = np.linspace(0, 10, Nx + 1)
y_edges_linear = np.linspace(-0.05, 0.05, Ny + 1)

# MANUALLY TRANSFORM the Y edges for the plot using arcsinh
y_edges_asinh = np.arcsinh(y_edges_linear / linear_width)

X_edges, Y_edges_asinh = np.meshgrid(x_edges, y_edges_asinh, indexing='ij')

# Generate Z data as a reshaped arange to easily see the Y-axis stretch/compression
Z = np.arange(Nx * Ny).reshape((Nx, Ny))

# 3. Create Custom Y Axis and Plot Widget
asinh_y_axis = AsinhAxisItem(linear_width=linear_width, orientation='left')

plot_widget = pg.PlotWidget(axisItems={'left': asinh_y_axis})
# Prevent window from flashing/showing on screen in interactive environments
plot_widget.setAttribute(QtCore.Qt.WidgetAttribute.WA_DontShowOnScreen, True)

plot_widget.setLabel('bottom', 'X Axis (Linear)')
plot_widget.setLabel('left', 'Y Axis (asinh scaled)')

# 4. Create PColorMesh
cmap = pg.colormap.get('viridis')
mesh = pg.PColorMeshItem(X_edges, Y_edges_asinh, Z, colorMap=cmap)
plot_widget.addItem(mesh)

# 5. Export
plot_widget.resize(800, 600)
app.processEvents()

exporter = pg.exporters.ImageExporter(plot_widget.plotItem)
output_filename = "pyqtgraph_pcolormesh_asinh_arange.png"
exporter.export(output_filename)

print(f"Plot successfully saved to {output_filename}")

K-means clustering: frequency, time, and amplitude features ?
or mass, tc, distance (and maybe other) params as the features, and the "distance between points" is sqrt(amp^2 + freq^2 + t^2)

In [ ]:
# Why does this code print a space after the first square bracket? but if i do axis = 1, it doesn't?
#a = np.array([[1, 2], [3, 4]]) - np.array([[10, 13], [5, -1]])
#np.sum(a**2, axis = 0)


a = np.array([[1, 2], [3, 4]]) - np.array([[10, 13], [5, -1]])
b = np.sum(a**2, axis = 1)
np.sqrt(b)

- skipping every other time in the time array? Robbie's answer: there was old data where pairs of 2 times basically gave the same value, so you could basically just take every other time to get better results. we shouldn't use this now.
- f or t coordinate array to wdm map correspondence ? Robbie's answer: the coordinates of the middle of the rectangles are the relevant values. (though he also said each coord could be off by up to a half rectangle? not sure)
- modes of the single BH? Robbie's answer: doesnt know what modes there are but the 2, 2 mode is the brightest one and you can just test different combos of modes to recreate/compare. also, the number of lines you see in the chirp signal in the spectrogram is basically the number of modes.
- ~a way to get fmin, fmax, and tmin when there are multiple chirps in one image?~

Training data creation "workflow":
- Each image will have 3-6 BHBs.
- We will create 100 images to start.
- For each of the 100 images, run this function (can be run using parallel processing):
    - Choose a random number (n) of BHBs, and for each BHB just random params.
    - Create a single total time domain waveform (TDWF) (2y) with all n BHBs, and also create a padded waveform (2y) for each BHB.
    - Input each TDWF into a function that gives you the WDM map, abs-valued.
    - Input the total TDWF into a function that makes the pcolormesh; save the figure without axes/labels/tickmarks/titles/etc. in YOLO26 directory format
    - Find the bounding boxes by inputting each WDM map into a function that takes in a WDM map, coalescence time, a threshold, observation time size, min search frequency, and top boundary location, and returns a list of boundary values.
    - Save the bounding box locations in YOLO26 directory format (YAML or whatever)
    - Delete the TDWFs, WDM maps, and bounding box arrays to save memory and prevent the kernel from dying.

In [ ]:
import numpy as np

def convertToYoloCoords(bboxIndices, tCoords, fCoords, yLimMin, yLimMax, asinhLinearWidth = 0.0005):
    """
    Converts matrix index boundaries into YOLO normalized coordinates,
    accounting for an asinh scaled y-axis. Made by Gemini.

    bboxIndices: tuple of (leftBoundary, rightBoundary, bottomBoundary, topBoundary)
    tCoords: 1D array of time coordinates
    fCoords: 1D array of frequency coordinates
    yLimMin: The lower limit of the matplotlib y-axis (e.g., 1e-8)
    yLimMax: The upper limit of the matplotlib y-axis (e.g., fmax)
    asinhLinearWidth: The linear_width parameter used in ax.set_yscale
    """

    leftIdx, rightIdx, bottomIdx, topIdx = bboxIndices

    # 1. get the physical data values (seconds and hz)
    tLeft = tCoords[leftIdx]
    tRight = tCoords[rightIdx]
    fBottom = fCoords[bottomIdx]
    fTop = fCoords[topIdx]

    # 2. handle the linear x-axis (time)
    tMin = tCoords[int(False)]
    tMax = tCoords[-1]

    xLeftNorm = (tLeft - tMin) / (tMax - tMin)
    xRightNorm = (tRight - tMin) / (tMax - tMin)

    # 3. handle the non-linear asinh y-axis (frequency)
    def asinhTransform(val):
        return np.arcsinh(val / asinhLinearWidth)

    sMin = asinhTransform(val = yLimMin)
    sMax = asinhTransform(val = yLimMax)

    yBottomNorm = (asinhTransform(val = fBottom) - sMin) / (sMax - sMin)
    yTopNorm = (asinhTransform(val = fTop) - sMin) / (sMax - sMin)

    # 4. invert the y-axis for yolo (yolo origin is top-left, matplotlib is bottom-left)
    yoloTop = 1.0 - yTopNorm
    yoloBottom = 1.0 - yBottomNorm

    # 5. calculate final yolo parameters
    xCenter = (xLeftNorm + xRightNorm) / 2.0
    yCenter = (yoloTop + yoloBottom) / 2.0
    width = xRightNorm - xLeftNorm
    height = yoloBottom - yoloTop

    # safely clip values between 0.0 and 1.0 to prevent out-of-bounds yolo errors
    xCenter = np.clip(a = xCenter, a_min = 0.0, a_max = 1.0)
    yCenter = np.clip(a = yCenter, a_min = 0.0, a_max = 1.0)
    width = np.clip(a = width, a_min = 0.0, a_max = 1.0)
    height = np.clip(a = height, a_min = 0.0, a_max = 1.0)

    return f'0 {xCenter:.6f} {yCenter:.6f} {width:.6f} {height:.6f}'



# Example usage in your loop:
# yLimMin = 1e-8  # from your ax.set_ylim(1e-8, fmax)
# yLimMax = fmax
#
# for idx, bbox in enumerate(BBoxes):
#     yoloString = convertToYoloCoords(bbox, tCoords, fCoords, yLimMin, yLimMax)
#     txtFileStr += f'{yoloString}\n'

In [ ]:
from multiprocessing import Pool
from pathlib import Path
import yaml

# TODO: Make checks for the 1st list item being < the 2nd item and provide useful messages if not; also check that tC is within the correct limits considering TObsTot and also TObs is in the correct limits, and all other variables (for example, beta and lamda (kinda; they repeat so idk) have min and max possible values; also, too high mass ratio might be bad). Edit: too high mass ratio was too bad.. doing 10:1 max lol; also, we should make it so that we clip the noise in case it gets too high, bc its possible it gives us a very large value even if it's improbable
def makeTrainingImage(
    numBHBsRange = [5, 20],
    # m1Range = [1e3, 1e7],
    # m2Range = [1e3, 1e7],
    logmTRange = [4,8],
    qRange = [0.1, 0.99999],
    chi1Range = [-0.9999999, 0.9999999],
    chi2Range = [-0.9999999, 0.9999999],
    distanceRange = [1e2, 1e5],
    cosincRange = [-1.0,1.0],
    phiRefRange = [0, 2*np.pi],
    psiRange = [0, 2 * np.pi],
    fMinRange = [1e-8, 1e-8],
    fRefRange = [1e-8, 1e-8],
    sinbetaRange = [-1.0,1.0],
    lambdaRange = [0.0, 2 * np.pi],
    tCRange = [0, 2 * ASTRONOMICAL_YEAR],
    dt = 2.5,
    waveletDuration = 3600 * 3,
    filename = 'WDMImage',
    imageFolder = Path('imageData') / 'images' / 'train',
    showBBoxSpectrogram = False,
    vmin = 1e-8,
    fmin = 1e-8,
    noiseLevel = 10**(-22.5),
    TObs = 5 * ASTRONOMICAL_YEAR / 12,
    TObsTot = 2 * ASTRONOMICAL_YEAR,
    allModes = False,
    responseFunc = None,
    multiprocessing = False):

    # Step 1: Make all the random params ------------------------------------------------------------------------------------------------------------------

    rng = np.random.default_rng()

    numBHBs = rng.integers(numBHBsRange[0], numBHBsRange[1], endpoint = True)
    tCs = rng.uniform(tCRange[0], tCRange[1], numBHBs)

    randomArgsDict = {}
    rangesList = [logmTRange, qRange, chi1Range, chi2Range, distanceRange, cosincRange, phiRefRange, psiRange, fMinRange, fRefRange, sinbetaRange, lambdaRange]
    randomParamNameList = ['logmT', 'q', 'chi1', 'chi2', 'distance', 'cosinc', 'phi_ref', 'psi', 'f_min', 'f_ref', 'sinbeta', 'lam']  # Coalescence time isn't used for the waveform or response gen
    for idx, paramName in enumerate(randomParamNameList):
        randomArgsDict[paramName] = rng.uniform(rangesList[idx][0], rangesList[idx][1], numBHBs)

    # Force m1 to be > m2 (for example, BBHx requires m1 to be > m2)
    #for idx in range(numBHBs):
    #    if randomArgsDict['m2'][idx] > randomArgs:
    #


    # Step 2: Get numBHBs waveforms -----------------------------------------------------------------------------------------------------------------------

    waveforms = []

    for idx in range(numBHBs):
        argSetDict = {paramName: paramVal[idx] for paramName, paramVal in randomArgsDict.items()}
        waveforms.append(getResponse(
            **argSetDict,
            dt = dt,
            Tobs = TObs,
            allModes = allModes,
            responseFunc = responseFunc
            )
        )


    # Step 3: Place the numBHBs waveforms into a single time array and in their own (padded) arrays -------------------------------------------------------

    # Total time series
    numTotTimes = int(TObsTot / dt)
    totTimeSeries = np.zeros(numTotTimes)

    for idx, wf in enumerate(waveforms):  # Can also do a list comprehension for a one-liner
        TDIPlacement(totTimeSeries, tCs[idx], wf[0], dt, plot = False)  # wf[0] to get A or X (I still don't know which it is but whatever)

    # Individual padded time series
    indivTimeSeries = np.zeros((numBHBs, numTotTimes))
    for idx, wf in enumerate(waveforms):  # Can also do a list comprehension for a one-liner
        TDIPlacement(indivTimeSeries[idx], tCs[idx], wf[0], dt, plot = False)


    # Step 4: Make all the WDM matrices -------------------------------------------------------------------------------------------------------------------

    # Get nT, nF, nD
    nT = int(TObsTot / waveletDuration)
    nD = len(totTimeSeries)
    nF = int(nD / nT)

    # Force nF and nT to be even because WDMWaveletTransforms says it assumes they're even and if they're not then results can be inaccurate
    if nF % 2 != 0:
        nF -= 1
    if nT % 2 != 0:
        nT -= 1

    nD = nF * nT
    
    # Total time series WDM map
    masterWDM = getWDMMatrix(totTimeSeries, nT, nF, noiseLevel = noiseLevel)
    
    # Individuals
    WDMs = [getWDMMatrix(timeSeries, nT, nF, noiseLevel = 1e-31) for timeSeries in indivTimeSeries]


    # Step 5: Make the spectrogram image of the total time series -----------------------------------------------------------------------------------------

    _, _, tCoords, fCoords = getSpectrogram(masterWDM, dt, vmin = vmin, showPlot = False, filename = filename, imageFolder = imageFolder)


    # Step 6: Get the bounding boxes for each BHB (iff we're saving the spectrogram image) ----------------------------------------------------------------

    if filename is not None:
        BBoxes = []

        fCoordSpacing = 1 / waveletDuration / 2  # 1 / WAVELET_DURATION for frequency of wavelet; / 2 for Nyquist. Can also do fCoords[-1] / len(fCoords).
        for idx, WDM in enumerate(WDMs):
            mT = 10**(randomArgsDict['logmT'][idx])
            m1 = mT/(randomArgsDict['q'][idx] + 1)
            m2 = mT - m1
            fISCO = 4400 / (m1 + m2)
            #BBoxes.append(getBoundingBox(WDM, round(tCs[idx] / waveletDuration), threshold = 0.2, tObsIdxSize = int(TObs / waveletDuration), fMinIdx = 0, manualTopBoundaryY = round(fISCO * 2 / fCoordSpacing), logNormalize = True))
            BBoxes.append(getBoundingBox(WDM, round(tCs[idx] / waveletDuration), topThreshold = 0.5, leftThreshold = 0.1, tObsIdxSize = int(TObs / waveletDuration), fMinIdx = 0, logNormalize = True))


        testTrainVal = Path(imageFolder).name
        with (Path(imageFolder).parent.parent / "labels" / testTrainVal / (filename + ".txt")).open('w') as file:  # TODO: maybe make this a bit nicer with relpath or something rather than parent.parent and whatever
            txtFileStr = ""
            for bbox in BBoxes:
                yoloString = convertToYoloCoords(bbox, tCoords, fCoords, 1e-8, fCoords[-1])
                txtFileStr += f'{yoloString}\n'
            txtFileStr = txtFileStr.removesuffix('\n')

            file.write(txtFileStr)

    # Step 7: Save the params for each BHB (iff we're saving the spectrogram image) -----------------------------------------------------------------------
    # Made by me and modified by Gemini

    if filename is not None:
        randomArgsDict['tC'] = tCs

    # convert numpy arrays to lists for safe yaml serialization
    yamlDict = {}
    for key, val in randomArgsDict.items():
        if isinstance(val, np.ndarray):
            yamlDict[key] = val.tolist()
        else:
            yamlDict[key] = val

    testTrainVal = Path(imageFolder).name
    outPath = Path(imageFolder).parent.parent / "params" / testTrainVal / (filename + ".yaml")

    with outPath.open(mode = 'w') as file:
        yaml.dump(data = yamlDict, stream = file, default_flow_style = False)


    # Step 8: Return WDMs/Time Series ---------------------------------------------------------------------------------------------------------------------

    return masterWDM, totTimeSeries, WDMs, indivTimeSeries


In [ ]:
%%time
masterWDM, totTimeSer, WDMs, indivTimeSer = makeTrainingImage(numBHBsRange = [1, 1], vmin = 1e-6)

In [ ]:
getSpectrogram(WDMs[2], dt = 2.5, cmap = 'Dark2', plotFeatures = True)

In [ ]:
import os
import math
import gc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from multiprocessing import Pool
from lisaconstants import ASTRONOMICAL_YEAR

# global variable for the worker processes to hold the c-backend wrapper
workerResponseFunc = None

def initWorker(dt, tObs):
    """
    initializes the response function once per cpu core.
    """
    global workerResponseFunc
    workerResponseFunc = instantiateResponseFunc(dt = dt, Tobs = tObs, allModes = False)

def generateSingleTask(args):
    """
    wrapper to execute the image generation with robust retry logic and aggressive cleanup.
    """
    fileName, folderPath = args
    
    dtConst = 2.5
    tObsConst = 5 * ASTRONOMICAL_YEAR / 12
    
    # retry loop -----------------------------------------------------------------
    success = False
    while not success:
        try:
            makeTrainingImage(
                filename = fileName,
                imageFolder = folderPath,
                dt = dtConst,
                TObs = tObsConst,
                responseFunc = workerResponseFunc,
                vmin = 1e-6
            )
            success = True
        except Exception as e:
            print(f'makeTrainingImage failed: {e}')
        finally:
            # strictly required to prevent matplotlib from hoarding ram across loops
            plt.close(fig = 'all')
            gc.collect()

def getOptimalCoreCount(ramPerTaskGb = 8.0):
    """
    dynamically calculates the maximum number of cores to use based on physical ram.
    prevents ssd swap thrashing which severely degrades performance.
    """
    posOne = int(True)
    
    try:
        # read total physical ram in bytes (works natively on macos/linux)
        bytesRam = os.sysconf(name = 'SC_PAGE_SIZE') * os.sysconf(name = 'SC_PHYS_PAGES')
        gbRam = bytesRam / (1024 ** 3)
        
        # calculate how many tasks fit in memory, leaving a few gb for the os
        availableGb = max(0, gbRam - 4.0)
        maxCoresByRam = max(posOne, int(availableGb / ramPerTaskGb))
        
        # use the maximum safe cores, but don't exceed actual cpu cores
        return min(os.cpu_count(), maxCoresByRam)
    except Exception:
        # fallback if sysconf fails
        return 2

def generateDataset(totalImages, baseDir = "imageData"):
    """
    calculates splits, sets up directories, and launches the optimized pool.
    """
    # split calculations ---------------------------------------------------------
    numTrain = math.floor(totalImages * 0.70)
    numVal = math.floor(totalImages * 0.20)
    numTest = totalImages - numTrain - numVal
    
    basePath = Path(baseDir)
    splits = ['train', 'val', 'test']
    counts = [numTrain, numVal, numTest]
    
    tasks = []
    
    # directory setup and task queueing ------------------------------------------
    for splitIdx in range(len(splits)):
        splitName = splits[splitIdx]
        splitCount = counts[splitIdx]
        
        imagePath = basePath / 'images' / splitName
        labelPath = basePath / 'labels' / splitName
        paramPath = basePath / 'params' / splitName
        
        imagePath.mkdir(parents = True, exist_ok = True)
        labelPath.mkdir(parents = True, exist_ok = True)
        paramPath.mkdir(parents = True, exist_ok = True)
        
        for imgIdx in range(splitCount):
            fName = f"{splitName}_img_{imgIdx:04d}"
            tasks.append((fName, imagePath))
            
    # launch the dynamically optimized pool --------------------------------------
    optimalCores = getOptimalCoreCount(ramPerTaskGb = 8.0)
    tasksPerChildLimit = 1
    
    print(f"system optimizing: running {totalImages} images across {optimalCores} cores to maximize speed without swap thrashing...")
    
    dtConst = 2.5
    tObsConst = 5 * ASTRONOMICAL_YEAR / 12
    
    with Pool(processes = optimalCores, initializer = initWorker, initargs = (dtConst, tObsConst), maxtasksperchild = tasksPerChildLimit) as pool:
        pool.map(func = generateSingleTask, iterable = tasks)
        
    print("dataset generation complete.")

if __name__ == '__main__':
    generateDataset(totalImages = 100)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib as mpl
from PIL import Image
from pathlib import Path

def plotYoloImage(imagePathStr, bbox = True):
    """
    plots a single spectrogram and overlays the yolo bounding boxes.
    """
    zeroIdx = int(False)
    posOne = int(True)
    posTwo = 2
    posThree = 3
    posFour = 4

    imgPath = Path(imagePathStr)
    labelPath = imgPath.parent.parent.parent / "labels" / imgPath.parent.name / (imgPath.stem + ".txt")

    if not imgPath.exists():
        print("image not found.")
        return

    img = Image.open(fp = imgPath)
    imgW, imgH = img.size

    fig, ax = plt.subplots(figsize = (10, 6))
    ax.imshow(X = img)

    # Extract all 20 colors from the tab20 colormap
    large_palette = mpl.colormaps['tab20'].colors
    
    # Set it as your new global color cycle
    plt.rcParams['axes.prop_cycle'] = plt.cycler(color=large_palette)

    # overlay bounding boxes -----------------------------------------------------
    if bbox and labelPath.exists():
        with open(file = labelPath, mode = 'r') as file:
            lines = file.readlines()

            for idx, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) >= 5:
                    xCenter = float(parts[posOne]) * imgW
                    yCenter = float(parts[posTwo]) * imgH
                    width = float(parts[posThree]) * imgW
                    height = float(parts[posFour]) * imgH

                    topLeftX = xCenter - (width / 2.0)
                    topLeftY = yCenter - (height / 2.0)

                    bbox = patches.Rectangle(
                        xy = (topLeftX, topLeftY),
                        width = width,
                        height = height,
                        linewidth = 0.5,
                        edgecolor = 'C' + str(idx),
                        facecolor = 'none'
                    )
                    ax.add_patch(p = bbox)

    ax.set_title(label = f"YOLO Annotations: {imgPath.name}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def viewAllDatasetImages(datasetDirPath):
    """
    iterates through all images in a specified dataset directory and plots them.
    """
    dataDir = Path(datasetDirPath)

    # gather all common image formats --------------------------------------------
    allImages = list(dataDir.glob(pattern = "*.png")) + list(dataDir.glob(pattern = "*.jpg"))

    if not allImages:
        print("no images found in the specified directory.")
        return

    for imgFile in allImages:
        plotYoloImage(imagePathStr = str(imgFile))

In [ ]:
plotYoloImage(Path('imageDataTest') / 'images' / 'train' / 'train_img_0000.jpg')
plotYoloImage(Path('imageDataTest') / 'images' / 'train' / 'train_img_0001.jpg')

In [ ]:
viewAllDatasetImages('imageDataTest2/images/train')

- total mass: draw on log(mT) between 4 and 8 uniform, then take 10^that mass
- q = mass ratio; choose m1 and then choose q, then find m2 from that
- modes: start with (2, 2); we can try other modes later
- spins: btwn -1 and 1, uniform (don't include -1 or 1, do like -0.99 to 0.99)
- distance: uniform between 1e2/1e5 Mpc
- inclination: uniform between -1 and 1, then arccos of that number.
- phi: 0 to 2 pi; uniform
- lambda: uniform btwn 0 and 2 pi
- beta: uniform between -1 and 1; take arcsin of it. we want to make sure they are uniform per unit celestial sphere area and there's less area at the top/bottom of a sphere.
- psi: 0 to 2 pi, uniform
- rate: uniform between 5 and 20


Thu, Jul. 9
- setting A2 = 0 removed the artifacts at the bottom of the image, so they were coming from that part of the noise.
- adding the noise to the time series instead of the WDM matrix makes it so that the tails are much less visible.
- Robbie's "PSD whitening" is to make it so that if you just plotted the noise it'd look flat. But adding the time domain noise to the time series data and then whitening with the PSD of the noise just cancels each other out so there's not much of a point. Anyway, the issue of the long tails is already solved so there's no need to have A2 nonzero.
- with regular priors, if i increase noise the tails are more comparable to Robbie's images (i.e. the "actual" BHBs), but then not all the BHBs are visible enough. maybe?

Fri, Jul. 10
- 3 variables that affect bounding box selection: vmin in the logNorm, leftThreshold, and noise level.

In [ ]:
wt, wf, wPSD = scalogram(A, dt, Nt=NT, Nf=NF, whiten_f = fest, whiten_PSD=est_psdA, log=True, logf=True, fmin = 1e-4, fmax = 1e-2, filename="mojito_sgram10.png", colormap = 'inferno')


In [ ]:
wavelet_specgram(wt, wf, wPSD, fmin=1e-4, fmax=1e-2,log=True,logf=True,filename="mojito_website.png", vmin=1e-4, vmax = 2e-1, cmap='inferno')

In [ ]:
print(NT, NF)

In [ ]:
waveletSpectrogram(A, dt, Nt=NT, Nf=NF, filename="sgramTest.jpg", vmin = 1e-4, vmax = 2e-1, fmin = 1e-4, fmax = 1e-2, noiseLevel = 0, whiten_f = fest, whiten_PSD = est_psdA)
